In [5]:
pip install opendatasets

In [6]:
from sklearn.linear_model import LinearRegression
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import opendatasets as od
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,root_mean_squared_error
warnings.filterwarnings("ignore")

In [7]:
data = od.download("https://www.kaggle.com/datasets/shree0910/ai-and-data-science-job-market-dataset-20202026/code")
data

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: gouravsinghthakur
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/shree0910/ai-and-data-science-job-market-dataset-20202026


100%|██████████| 184k/184k [00:00<00:00, 289MB/s]

In [8]:
Data = pd.read_csv("/content/ai-and-data-science-job-market-dataset-20202026/AI Job Market Dataset.csv")
Data.head()

,job_id,job_title,company_size,company_industry,country,remote_type,experience_level,years_experience,education_level,skills_python,skills_sql,skills_ml,skills_deep_learning,skills_cloud,salary,job_posting_month,job_posting_year,hiring_urgency,job_openings
0,1,AI Engineer,Startup,Retail,Canada,Remote,Senior,2,Master,0,0,0,1,0,158322,6,2024,Low,4
1,2,Machine Learning Engineer,MNC,Technology,Australia,Hybrid,Mid,0,Bachelor,1,1,1,0,1,163666,11,2026,High,9
2,3,Machine Learning Engineer,MNC,Technology,Germany,Onsite,Mid,14,Master,1,0,1,0,1,158556,3,2026,High,9
3,4,Business Analyst,Startup,Healthcare,Germany,Remote,Mid,9,Master,0,1,0,1,1,95775,3,2025,High,7
4,5,Data Scientist,MNC,Healthcare,Germany,Hybrid,Mid,5,Master,1,1,1,0,0,111873,12,2021,Low,2


In [9]:
X = Data.drop(columns=["salary"])
y = Data["salary"]

In [18]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(include=["int64","float64"]).columns.tolist()

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)
preprocessor

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['job_id', 'years_experience', 'skills_python',
                                  'skills_sql', 'skills_ml',
                                  'skills_deep_learning', 'skills_cloud',
                                  'job_posting_month', 'job_posting_year',
                                  'job_openings']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['job_title', 'company_size',
                                  'company_industry', 'country', 'remote_type',
                                  'experience_level', 'education_level',
                                  'hiring_urgency'])])

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [23]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(),
    "XGBoost": XGBRegressor(random_state=42),
    "LightGBM": LGBMRegressor(force_row_wise=True,random_state=42)
}

In [24]:
results = []

for name, model in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    preds = pipe.predict(X_test)

    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    results.append({
        "Model": name,
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse
    })

[LightGBM] [Info] Total Bins 387
[LightGBM] [Info] Number of data points in the train set: 8276, number of used features: 45
[LightGBM] [Info] Start training from score 113711.801957


In [33]:
results_df = pd.DataFrame(results)

results_df.sort_values("R2", ascending=False).round(4)

,Model,R2,MAE,RMSE
0,Linear Regression,0.9919,2448.5286,2846.9065
4,LightGBM,0.9916,2468.1335,2889.0829
2,Gradient Boosting,0.9915,2482.0610,2903.0937
1,Random Forest,0.9910,2543.0989,2993.8194
3,XGBoost,0.9906,2562.1707,3062.3509


In [34]:
best_model = results_df.sort_values("R2", ascending=False).iloc[0]["Model"]

print("Best model:", best_model)

Best model: Linear Regression


In [35]:
final_model = models[best_model]
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_model)
])
final_pipeline.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['job_id', 'years_experience',
                                                   'skills_python',
                                                   'skills_sql', 'skills_ml',
                                                   'skills_deep_learning',
                                                   'skills_cloud',
                                                   'job_posting_month',
                                                   'job_posting_year',
                                                   'job_openings']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job_title', 'company_size',
                                                   'company_industry',
                                                   'country', 'remote_type',
                                                   'experience_level',
                                                   'education_level',
                                                   'hiring_urgency'])])),
                ('model', LinearRegression())])

In [ ]:
joblib.dump(final_pipeline, "models/salary_prediction_model.pkl")

['salary_prediction_model.pkl']